In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-stage-3-2026")

print("Path to dataset files:", path)

In [ ]:
# Write your code here

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np
import os

# Define transforms for training and testing
train_transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.RandomRotation(15),  # Augmentation
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

test_transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Load datasets using ImageFolder
data_path = "/kaggle/input/q1-stage-3-2026"

# Find the train and test directories
train_dir = os.path.join(data_path, 'train')
test_dir = os.path.join(data_path, 'test')

# If the structure is different, adjust accordingly
if not os.path.exists(train_dir):
    # Try to find the actual structure
    for root, dirs, files in os.walk(data_path):
        if 'train' in dirs:
            train_dir = os.path.join(root, 'train')
            test_dir = os.path.join(root, 'test')
            break

train_dataset = datasets.ImageFolder(root=train_dir, transform=train_transform)
test_dataset = datasets.ImageFolder(root=test_dir, transform=test_transform)

# Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2)

# Get class names
class_names = train_dataset.classes
num_classes = len(class_names)

print(f"Training samples: {len(train_dataset)}")
print(f"Testing samples: {len(test_dataset)}")
print(f"Number of classes: {num_classes}")
print(f"Classes: {class_names}")
print(f"Number of training batches: {len(train_loader)}")
print(f"Number of testing batches: {len(test_loader)}")

# Display sample images
def imshow(img, title=None):
    """Display image with denormalization"""
    img = img.numpy().transpose((1, 2, 0))
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    img = std * img + mean
    img = np.clip(img, 0, 1)
    return img

# Get a batch of training data
dataiter = iter(train_loader)
images, labels = next(dataiter)

# Display 8 samples
fig, axes = plt.subplots(2, 4, figsize=(15, 7))
for i, ax in enumerate(axes.flat):
    img = imshow(images[i])
    ax.imshow(img)
    ax.set_title(f'Class: {class_names[labels[i]]}')
    ax.axis('off')
plt.tight_layout()
plt.show()


In [ ]:
# Write your code here

class PotatoCNN(nn.Module):
    def __init__(self, num_classes=3):
        super(PotatoCNN, self).__init__()

        # Convolutional Layer 1
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.relu1 = nn.ReLU()
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)  # 32x32 -> 16x16

        # Convolutional Layer 2
        self.conv2 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        self.relu2 = nn.ReLU()
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)  # 16x16 -> 8x8

        # Convolutional Layer 3
        self.conv3 = nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(128)
        self.relu3 = nn.ReLU()
        self.pool3 = nn.MaxPool2d(kernel_size=2, stride=2)  # 8x8 -> 4x4

        # Convolutional Layer 4
        self.conv4 = nn.Conv2d(in_channels=128, out_channels=256, kernel_size=3, padding=1)
        self.bn4 = nn.BatchNorm2d(256)
        self.relu4 = nn.ReLU()
        self.pool4 = nn.MaxPool2d(kernel_size=2, stride=2)  # 4x4 -> 2x2

        # Convolutional Layer 5
        self.conv5 = nn.Conv2d(in_channels=256, out_channels=512, kernel_size=3, padding=1)
        self.bn5 = nn.BatchNorm2d(512)
        self.relu5 = nn.ReLU()
        self.pool5 = nn.MaxPool2d(kernel_size=2, stride=2)  # 2x2 -> 1x1

        # Fully Connected Layers
        self.fc1 = nn.Linear(512 * 1 * 1, 256)
        self.dropout1 = nn.Dropout(0.5)
        self.fc2 = nn.Linear(256, 128)
        self.dropout2 = nn.Dropout(0.5)
        self.fc3 = nn.Linear(128, num_classes)

    def forward(self, x):
        # Conv Block 1
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu1(x)
        x = self.pool1(x)

        # Conv Block 2
        x = self.conv2(x)
        x = self.bn2(x)
        x = self.relu2(x)
        x = self.pool2(x)

        # Conv Block 3
        x = self.conv3(x)
        x = self.bn3(x)
        x = self.relu3(x)
        x = self.pool3(x)

        # Conv Block 4
        x = self.conv4(x)
        x = self.bn4(x)
        x = self.relu4(x)
        x = self.pool4(x)

        # Conv Block 5
        x = self.conv5(x)
        x = self.bn5(x)
        x = self.relu5(x)
        x = self.pool5(x)

        # Flatten
        x = x.view(x.size(0), -1)

        # Fully Connected Layers
        x = self.fc1(x)
        x = self.dropout1(x)
        x = nn.ReLU()(x)

        x = self.fc2(x)
        x = self.dropout2(x)
        x = nn.ReLU()(x)

        x = self.fc3(x)

        return x


# Create model instance
model = PotatoCNN(num_classes=num_classes)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print("\nModel Architecture:")
print(model)


In [ ]:
# Write your code here

def train_one_epoch(model, train_loader, criterion, optimizer, device):
    """
    Train the model for one epoch.

    Returns:
        avg_loss: Average training loss
        accuracy: Training accuracy
    """
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        # Zero the gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(images)
        loss = criterion(outputs, labels)

        # Backward pass and optimize
        loss.backward()
        optimizer.step()

        # Statistics
        running_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    avg_loss = running_loss / total
    accuracy = 100 * correct / total

    return avg_loss, accuracy


def validate(model, test_loader, criterion, device):
    """
    Validate the model.

    Returns:
        avg_loss: Average validation loss
        accuracy: Validation accuracy
    """
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            labels = labels.to(device)

            # Forward pass
            outputs = model(images)
            loss = criterion(outputs, labels)

            # Statistics
            running_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    avg_loss = running_loss / total
    accuracy = 100 * correct / total

    return avg_loss, accuracy


print("Training and validation functions defined successfully!")

In [ ]:
# Setup device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Move model to device
model = model.to(device)

# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Learning rate scheduler (verbose)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

# Training parameters
num_epochs = 25

# Lists to store metrics
train_losses = []
val_losses = []
train_accs = []
val_accs = []

# Training loop
print("\nStarting training...\n")
best_val_acc = 0.0

for epoch in range(num_epochs):
    # Train
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)

    # Validate
    val_loss, val_acc = validate(model, test_loader, criterion, device)

    # Step the scheduler
    old_lr = optimizer.param_groups[0]['lr']
    scheduler.step(val_loss)
    new_lr = optimizer.param_groups[0]['lr']

    # Store metrics
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accs.append(train_acc)
    val_accs.append(val_acc)

    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), 'best_potato_cnn.pth')

    # Print progress
    print(f"Epoch [{epoch+1}/{num_epochs}]")
    print(f"  Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%")
    print(f"  Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc:.2f}%")
    if new_lr < old_lr:
        print(f"  Learning rate reduced: {old_lr:.6f} → {new_lr:.6f}")
    print()

print("Training completed!")
print(f"Best Validation Accuracy: {best_val_acc:.2f}%")

In [ ]:
# Plot training and validation metrics
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Loss plot
epochs_range = range(1, num_epochs + 1)
ax1.plot(epochs_range, train_losses, 'b-o', label='Training Loss', linewidth=2)
ax1.plot(epochs_range, val_losses, 'r-o', label='Validation Loss', linewidth=2)
ax1.set_xlabel('Epoch', fontsize=12)
ax1.set_ylabel('Loss', fontsize=12)
ax1.set_title('Training and Validation Loss', fontsize=14, fontweight='bold')
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)

# Accuracy plot
ax2.plot(epochs_range, train_accs, 'b-o', label='Training Accuracy', linewidth=2)
ax2.plot(epochs_range, val_accs, 'r-o', label='Validation Accuracy', linewidth=2)
ax2.set_xlabel('Epoch', fontsize=12)
ax2.set_ylabel('Accuracy (%)', fontsize=12)
ax2.set_title('Training and Validation Accuracy', fontsize=14, fontweight='bold')
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nFinal Results:")
print(f"Training Loss: {train_losses[-1]:.4f}")
print(f"Training Accuracy: {train_accs[-1]:.2f}%")
print(f"Validation Loss: {val_losses[-1]:.4f}")
print(f"Validation Accuracy: {val_accs[-1]:.2f}%")

In [ ]:
# Write your code here

class PotatoCNN_Residual(nn.Module):
    def __init__(self, num_classes=3):
        super(PotatoCNN_Residual, self).__init__()

        # Convolutional Layer 1
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.relu1 = nn.ReLU()
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)  # 32x32 -> 16x16

        # Convolutional Layer 2 (Skip connection starts here)
        self.conv2 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        self.relu2 = nn.ReLU()
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)  # 16x16 -> 8x8

        # Convolutional Layer 3
        self.conv3 = nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(128)
        self.relu3 = nn.ReLU()
        self.pool3 = nn.MaxPool2d(kernel_size=2, stride=2)  # 8x8 -> 4x4

        # Convolutional Layer 4 (Skip connection ends here)
        self.conv4 = nn.Conv2d(in_channels=128, out_channels=256, kernel_size=3, padding=1)
        self.bn4 = nn.BatchNorm2d(256)
        self.relu4 = nn.ReLU()
        self.pool4 = nn.MaxPool2d(kernel_size=2, stride=2)  # 4x4 -> 2x2

        # Skip connection adaptation
        # We need to match dimensions from layer2 (64 channels, 8x8) to layer4 input (128 channels, 4x4)
        self.skip_conv = nn.Conv2d(in_channels=64, out_channels=128, kernel_size=1)
        self.skip_pool = nn.MaxPool2d(kernel_size=2, stride=2)  # 8x8 -> 4x4

        # Convolutional Layer 5
        self.conv5 = nn.Conv2d(in_channels=256, out_channels=512, kernel_size=3, padding=1)
        self.bn5 = nn.BatchNorm2d(512)
        self.relu5 = nn.ReLU()
        self.pool5 = nn.MaxPool2d(kernel_size=2, stride=2)  # 2x2 -> 1x1

        # Fully Connected Layers
        self.fc1 = nn.Linear(512 * 1 * 1, 256)
        self.dropout1 = nn.Dropout(0.5)
        self.fc2 = nn.Linear(256, 128)
        self.dropout2 = nn.Dropout(0.5)
        self.fc3 = nn.Linear(128, num_classes)

    def forward(self, x):
        # Conv Block 1
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu1(x)
        x = self.pool1(x)

        # Conv Block 2 (Save for skip connection)
        x = self.conv2(x)
        x = self.bn2(x)
        x = self.relu2(x)
        x = self.pool2(x)
        skip = x  # Save skip connection (64 channels, 8x8)

        # Conv Block 3
        x = self.conv3(x)
        x = self.bn3(x)
        x = self.relu3(x)
        x = self.pool3(x)  # (128 channels, 4x4)

        # Adapt skip connection to match dimensions
        skip = self.skip_conv(skip)  # 64 -> 128 channels
        skip = self.skip_pool(skip)  # 8x8 -> 4x4

        # Add residual connection (summation)
        x = x + skip

        # Conv Block 4
        x = self.conv4(x)
        x = self.bn4(x)
        x = self.relu4(x)
        x = self.pool4(x)

        # Conv Block 5
        x = self.conv5(x)
        x = self.bn5(x)
        x = self.relu5(x)
        x = self.pool5(x)

        # Flatten
        x = x.view(x.size(0), -1)

        # Fully Connected Layers
        x = self.fc1(x)
        x = self.dropout1(x)
        x = nn.ReLU()(x)

        x = self.fc2(x)
        x = self.dropout2(x)
        x = nn.ReLU()(x)

        x = self.fc3(x)

        return x


# Create residual model instance
model_res = PotatoCNN_Residual(num_classes=num_classes)

# Count parameters
total_params_res = sum(p.numel() for p in model_res.parameters())
trainable_params_res = sum(p.numel() for p in model_res.parameters() if p.requires_grad)

print(f"Residual Model - Total parameters: {total_params_res:,}")
print(f"Residual Model - Trainable parameters: {trainable_params_res:,}")
print("\nResidual Model Architecture:")
print(model_res)


In [ ]:
# Train the residual model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_res = model_res.to(device)

# Define loss function and optimizer
criterion_res = nn.CrossEntropyLoss()
optimizer_res = optim.Adam(model_res.parameters(), lr=0.001)

# Learning rate scheduler (removed verbose)
scheduler_res = optim.lr_scheduler.ReduceLROnPlateau(optimizer_res, mode='min', factor=0.5, patience=3)

# Training parameters
num_epochs_res = 25

# Lists to store metrics
train_losses_res = []
val_losses_res = []
train_accs_res = []
val_accs_res = []

# Training loop
print("\nStarting training with Residual Connections...\n")
best_val_acc_res = 0.0

for epoch in range(num_epochs_res):
    # Train
    train_loss, train_acc = train_one_epoch(model_res, train_loader, criterion_res, optimizer_res, device)

    # Validate
    val_loss, val_acc = validate(model_res, test_loader, criterion_res, device)

    # Step the scheduler
    old_lr = optimizer_res.param_groups[0]['lr']
    scheduler_res.step(val_loss)
    new_lr = optimizer_res.param_groups[0]['lr']

    # Store metrics
    train_losses_res.append(train_loss)
    val_losses_res.append(val_loss)
    train_accs_res.append(train_acc)
    val_accs_res.append(val_acc)

    # Save best model
    if val_acc > best_val_acc_res:
        best_val_acc_res = val_acc
        torch.save(model_res.state_dict(), 'best_potato_cnn_residual.pth')

    # Print progress
    print(f"Epoch [{epoch+1}/{num_epochs_res}]")
    print(f"  Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%")
    print(f"  Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc:.2f}%")
    if new_lr < old_lr:
        print(f"  Learning rate reduced: {old_lr:.6f} → {new_lr:.6f}")
    print()

print("Training with Residual Connections completed!")
print(f"Best Validation Accuracy: {best_val_acc_res:.2f}%")

In [ ]:
# Plot residual model metrics
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Loss plot
epochs_range_res = range(1, num_epochs_res + 1)
ax1.plot(epochs_range_res, train_losses_res, 'b-o', label='Training Loss', linewidth=2)
ax1.plot(epochs_range_res, val_losses_res, 'r-o', label='Validation Loss', linewidth=2)
ax1.set_xlabel('Epoch', fontsize=12)
ax1.set_ylabel('Loss', fontsize=12)
ax1.set_title('Residual Model - Training and Validation Loss', fontsize=14, fontweight='bold')
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)

# Accuracy plot
ax2.plot(epochs_range_res, train_accs_res, 'b-o', label='Training Accuracy', linewidth=2)
ax2.plot(epochs_range_res, val_accs_res, 'r-o', label='Validation Accuracy', linewidth=2)
ax2.set_xlabel('Epoch', fontsize=12)
ax2.set_ylabel('Accuracy (%)', fontsize=12)
ax2.set_title('Residual Model - Training and Validation Accuracy', fontsize=14, fontweight='bold')
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Compare both models
print("\n" + "="*60)
print("COMPARISON: Base Model vs Residual Model")
print("="*60)
print(f"\nBase Model:")
print(f"  Best Validation Accuracy: {best_val_acc:.2f}%")
print(f"  Final Training Accuracy: {train_accs[-1]:.2f}%")
print(f"  Final Validation Accuracy: {val_accs[-1]:.2f}%")

print(f"\nResidual Model:")
print(f"  Best Validation Accuracy: {best_val_acc_res:.2f}%")
print(f"  Final Training Accuracy: {train_accs_res[-1]:.2f}%")
print(f"  Final Validation Accuracy: {val_accs_res[-1]:.2f}%")

print(f"\nImprovement: {best_val_acc_res - best_val_acc:.2f}%")
print("="*60)